# Images and Colors

The purpose of this assignment is to get practice with interactivity using the ipywidgets interface and to think carefully about how our data (and its types) can effect our design choices.

**Please see Homework Prompt in PrairieLearn interface for more details on the requirements for this assignment.**

A rough outline of elements of code and write-up is shown below:

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
df = pd.read_csv(url, parse_dates=["date"])
df.columns

Index(['observed', 'location_details', 'county', 'state', 'season', 'title',
       'latitude', 'longitude', 'date', 'number', 'classification', 'geohash',
       'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point',
       'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity',
       'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index',
       'visibility', 'wind_bearing', 'wind_speed', 'location'],
      dtype='object')

In [2]:
numeric_columns = list(df.select_dtypes(include=['number']).columns)
categorical_columns = list(df.select_dtypes(include=['object']).columns)
df.head(5)

,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaT,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaT,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


## Plot \#1 Code

In [3]:
valid_columns = ["state", "season", "classification", "temperature_high", "humidity", "wind_speed", "pressure"]

x_dropdown = widgets.Dropdown(options=valid_columns, description='X-axis:')
y_dropdown = widgets.Dropdown(options=valid_columns, description='Y-axis:')

marker_dropdown = widgets.Dropdown(options=['o', 's', '^', 'D'], description='Marker:')

color_dropdown = widgets.Dropdown(options=all_columns, description='Color by:')

def update_scatter(x, y, marker, color_by):
    plt.figure(figsize=(8, 6))
    
    if pd.api.types.is_numeric_dtype(df[x]):
        x_data = df[x]
    else:
        x_data = pd.factorize(df[x])[0]
    
    if pd.api.types.is_numeric_dtype(df[y]):
        y_data = df[y]
    else:
        y_data = pd.factorize(df[y])[0]
    
    if pd.api.types.is_numeric_dtype(df[color_by]):
        c_data = df[color_by]
        cmap = 'viridis'
    else:
        c_data = pd.factorize(df[color_by])[0]
        cmap = 'tab10'
    
    scatter = plt.scatter(x_data, y_data, marker=marker, c=c_data, cmap=cmap)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"Scatter Plot of {y} vs {x}")
    
    plt.colorbar(scatter, label=color_by)
    plt.show()

widgets.interact(update_scatter, x=x_dropdown, y=y_dropdown, marker=marker_dropdown, color_by=color_dropdown)

NameError: name 'all_columns' is not defined

## Plot \#1 Write Up

This plot is an interactive scatter plot that lets the user pick two columns to compare, change the marker type, and color the points based on another variable. At first, I let the user pick any column from the dataset, but I then ran into issues with long text fields like `"observed"` and `"title"`, causing errors and made the plot unreadable.

To fix this, I limited the column choices to some that make sense for a scatter plot:  
- **Categorical columns**: `"state"`, `"season"`, `"classification"`  
- **Numerical columns**: `"temperature_high"`, `"humidity"`, `"wind_speed"`, `"pressure"`  

Users can experiment and match different variables to see how they relate, also noting that I used python's "factorize" function to appropriately numerize categorical values. The color and marker choices also help highlight patterns, making it easier to explore the data visually.

## Plot \#2 Code

In [4]:
bin_field_dropdown = widgets.Dropdown(options=valid_columns, description='Bin Field:')

bin_slider = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Bins:')

def update_histogram(bin_field, bins):
    plt.figure(figsize=(8, 6))

    if pd.api.types.is_numeric_dtype(df[bin_field]):
        plt.hist(df[bin_field].dropna(), bins=bins, edgecolor='black')
        plt.xlabel(bin_field)
        plt.ylabel('Frequency')
        plt.title(f"Histogram of {bin_field} with {bins} bins")

    else:
        counts = df[bin_field].value_counts()

        plt.bar(counts.head(10).index.astype(str), counts.head(10).values, edgecolor='black')
        plt.xlabel(bin_field)
        plt.ylabel('Count')
        plt.title(f"Bar Chart of {bin_field}")
        plt.xticks(rotation=45)

    plt.show()

widgets.interact(update_histogram, bin_field=bin_field_dropdown, bins=bin_slider)

interactive(children=(Dropdown(description='Bin Field:', options=('state', 'season', 'classification', 'temper…

<function __main__.update_histogram(bin_field, bins)>

## Plot \#2 Write Up

This plot is an interactive histogram--for numerical data--and bar chart--for categorical data--that helps show how frequently different values appear in the dataset. Again, I initially allowed the user to pick any column, but just like with Plot #1, I ran into issues with long text fields like `"observed"` and `"title"`, which didn’t work well for plotting and caused errors, resulting to using the same columns used in Plot #1.

For bar charts, I also limit the display to the top 10 categories so the chart isn’t overcrowded. This makes it easier to see trends, like which states report the most sightings or what temperature ranges are most common in Bigfoot reports.